# Step 2 - Exploratory data analysis

**CSC3119 Assignment 4.** This notebook reads `outputs/01_prep/hourly_clean.parquet` and answers the questions the modelling step depends on:

* How is PM2.5 distributed, and what does the skew mean for model choice?
* Are the large values faults or real pollution events?
* Is there a daily cycle, and a seasonal level shift?
* What are the Z-score constants that define the modelling target?
* How strongly do temperature, humidity and PM10 relate to PM2.5?

Every figure is saved to `outputs/02_eda/` and `report/figures/` for the report, and the conclusions are written to `eda_meta.json` so Step 3 can read them back rather than re-deciding.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

PROJECT = Path.cwd()
while not (PROJECT / "src" / "aq_common.py").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / "src"))
import aq_common as A

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

A.set_stage("eda")
A.banner("STEP 2 / 6  -  Exploratory data analysis")
LOG = A.PipelineLog("step2_eda")


  STEP 2 / 6  -  Exploratory data analysis


2026-10-18 12:22:04 | INFO | === step2_eda ===


## The focal sensors
The brief asks for **about five Kampala sensors**. The choice is ours to defend, so we take the five city sensors with the most observed hours (deterministic, ties broken by name). Using the densest sensors keeps the EDA honest.

In [2]:
grid = pd.read_parquet(A.stage_dir("prep") / "hourly_clean.parquet")
PREP = A.load_json(A.stage_dir("prep") / "prep_meta.json")
obs = grid[grid["is_observed"]].copy()
LOG.log(f"Loaded hourly grid {grid.shape}; observed rows {len(obs):,}")

print(f"grid rows        : {len(grid):,}")
print(f"observed PM2.5   : {int(grid['is_observed'].sum()):,} "
      f"({100*grid['is_observed'].mean():.2f}% of sensor-hours)")
print(f"sensors / sites  : {obs['device_name'].nunique()} / {obs['site_name'].nunique()}")
print(f"networks         : {sorted(obs['network'].dropna().unique().tolist())}")
print(f"calendar days    : {sorted(str(d) for d in obs['datetime'].dt.date.unique())}")

# The five focal sensors: the city sensors with the most data.  Chosen once, here,
# so every figure and the report refer to the same five.
focal, focal_all = A.select_focal_sensors(obs, n=5)
FOCAL = focal["device_name"].tolist()
focal.to_csv(A.stage_dir("eda") / "focal_sensors.csv", index=False)
print("\nThe five focal Kampala sensors (most observed hours in greater Kampala):")
print(focal.to_string(index=False))
LOG.log("Focal sensors: " + ", ".join(FOCAL))

2026-10-18 12:22:05 | INFO | Loaded hourly grid (758112, 18); observed rows 14,272
grid rows        : 758,112
observed PM2.5   : 14,272 (1.88% of sensor-hours)
sensors / sites  : 149 / 145
networks         : ['airgradient', 'airqo']
calendar days    : ['2026-01-01', '2026-01-02', '2026-01-03', '2026-04-01', '2026-04-02', '2026-04-03', '2026-06-01', '2026-06-02', '2026-06-03']

The five focal Kampala sensors (most observed hours in greater Kampala):
    device_name  hours                     site_name     network  latitude  longitude
          aq_66    147       Nakasero II, Kampala 66       airqo  0.322310  32.575754
       aq_g5_37    147 Civic Centre, Kampala Central       airqo  0.318035  32.589921
          aq_47    145             Bukoto I, Kawempe       airqo  0.351654  32.591230
          aq_58    145     Nansana east ward, Wakiso       airqo  0.375750  32.527594
ag_588c81267534    142                   Salama Road airgradient  0.333540  32.568610
2026-10-18 12:22:05 | INFO | Fo

## 1. Distributions and skewness
A density plot shows how often each value occurs. PM2.5 is right-skewed: most hours are relatively clean, but a long tail reaches high concentrations. In a right-skewed variable the mean sits above the median because a few large values pull it up - so the median describes a *typical* hour better, while the tail is where the health risk lives.

In [3]:
print("\n" + "="*78)
print(" 1. DISTRIBUTIONS AND SKEWNESS")
print("="*78)

skew_rows = []
for dev in FOCAL:
    s = obs.loc[obs["device_name"] == dev, "pm25"].dropna()
    if len(s) < 5:
        continue
    skew_rows.append({"device_name": dev,
                      "n": len(s), "mean": s.mean(), "median": s.median(),
                      "std": s.std(), "skewness": stats.skew(s),
                      "p95": s.quantile(0.95), "max": s.max()})
skew = pd.DataFrame(skew_rows)
print(skew.round(2).to_string(index=False))
skew.to_csv(A.stage_dir("eda") / "distribution_summary.csv", index=False)
pooled_skew = stats.skew(obs["pm25"].dropna())
print(f"\nPooled PM2.5 skewness: {pooled_skew:.2f}  "
      f"(positive = right-skewed: a few polluted hours pull the mean above the median).")
print("DECISION: keep the tail (it is real pollution) and prefer models robust to")
print("extreme values. Report RMSE beside MAE so big misses stay visible.")

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
panels = [("pm25", "PM2.5 (ug/m3)"), ("pm10", "PM10 (ug/m3)"),
          ("temp", "Temperature (degC)"), ("humidity", "Humidity (%)")]
for ax, (col, label) in zip(axes.ravel(), panels):
    for dev in FOCAL:
        s = obs.loc[obs["device_name"] == dev, col].dropna()
        if len(s) > 5:
            sns.kdeplot(s, ax=ax, label=dev, lw=1.3, alpha=0.8)
    ax.set_xlabel(label)
    ax.set_ylabel("density")
    ax.set_title(label.split(" (")[0])
    ax.grid(alpha=0.3)
    ax.legend(fontsize=5, loc="upper right")   # each panel is independently readable
axes[0, 0].axvline(A.AQI_BANDS[1][1], color="grey", ls="--", lw=0.8)
fig.suptitle("Fig 2.1  Distributions of PM2.5, PM10, temperature and humidity (five focal sensors)",
             fontsize=12, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.97])
A.savefig(fig, "fig2_1_distributions.png")
print("saved fig2_1_distributions.png")


 1. DISTRIBUTIONS AND SKEWNESS


    device_name   n  mean  median   std  skewness    p95    max
          aq_66 147 37.13   30.65 24.70      1.06  73.51 142.44
       aq_g5_37 147 50.41   27.38 50.78      1.36 164.93 188.20
          aq_47 145 59.01   43.35 43.67      1.06 144.13 186.43
          aq_58 145 27.74   19.79 18.40      1.90  70.38 100.81
ag_588c81267534 142 71.91   52.86 53.32      1.65 180.70 290.98

Pooled PM2.5 skewness: 2.08  (positive = right-skewed: a few polluted hours pull the mean above the median).
DECISION: keep the tail (it is real pollution) and prefer models robust to
extreme values. Report RMSE beside MAE so big misses stay visible.


saved fig2_1_distributions.png


## 2. Box plots and outliers
A box plot shows the median, the middle 50% (Q1-Q3) and whiskers at 1.5*IQR. The robust Z-score (median/MAD) is an alternative that the outliers themselves cannot distort. **We flag outliers and keep them**: they cluster at the traffic/cooking peaks, so they are events, not faults.

In [4]:
print("\n" + "="*78)
print(" 2. BOX PLOTS: IQR vs ROBUST-Z OUTLIERS")
print("="*78)

rows = []
for dev in FOCAL:
    s = obs.loc[obs["device_name"] == dev, "pm25"].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    iqr_out = int(((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum())
    rz = A.robust_z(s)
    rz_out = int((rz.abs() > A.ROBUST_Z_THRESHOLD).sum())
    rows.append({"device_name": dev, "hours": len(s), "IQR_outliers": iqr_out,
                 "robustZ_outliers": rz_out, "max": s.max()})
box = pd.DataFrame(rows)
print(box.to_string(index=False))
box.to_csv(A.stage_dir("eda") / "outlier_counts_focal.csv", index=False)
print("\nIQR outliers use Q1/Q3 +/- 1.5*IQR; robust-Z uses median/MAD. The robust")
print("version is used downstream because the outliers themselves inflate the IQR.")
print("Treatment: FLAG, do not delete - the spikes occur at the traffic/cooking")
print("peaks, so they are events, not faults (only >500 ug/m3 was removed in Step 1).")

fig, ax = plt.subplots(figsize=(11, 4.5))
data = [obs.loc[obs['device_name'] == dev, "pm25"].dropna().values for dev in FOCAL]
bp = ax.boxplot(data, tick_labels=FOCAL, patch_artist=True, showfliers=True,
                flierprops=dict(marker=".", markersize=4, markerfacecolor="#c44e52",
                                markeredgecolor="none", alpha=0.5))
for patch, c in zip(bp["boxes"], sns.color_palette("deep", len(FOCAL))):
    patch.set_facecolor(c); patch.set_alpha(0.6)
ax.set_ylabel("PM2.5 (ug/m3)")
ax.set_title("Fig 2.2  PM2.5 box plots for the five focal sensors (dots = outliers)")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
A.savefig(fig, "fig2_2_boxplots.png")
print("saved fig2_2_boxplots.png")


 2. BOX PLOTS: IQR vs ROBUST-Z OUTLIERS
    device_name  hours  IQR_outliers  robustZ_outliers    max
          aq_66    147             1                 1 142.44
       aq_g5_37    147            17                25 188.20
          aq_47    145             6                 6 186.43
          aq_58    145            11                23 100.81
ag_588c81267534    142             5                 5 290.98

IQR outliers use Q1/Q3 +/- 1.5*IQR; robust-Z uses median/MAD. The robust
version is used downstream because the outliers themselves inflate the IQR.
Treatment: FLAG, do not delete - the spikes occur at the traffic/cooking
peaks, so they are events, not faults (only >500 ug/m3 was removed in Step 1).


saved fig2_2_boxplots.png


## 3. Time series with AQI bands
Each line is a focal sensor; the coloured bands are the AQI categories. A break in a line is an offline hour. This single plot exposes both the gaps and the seasonal level shift.

In [5]:
print("\n" + "="*78)
print(" 3. TIME SERIES WITH AQI BANDS")
print("="*78)

focal_grid = grid[grid["device_name"].isin(FOCAL)].copy()
ymax = float(np.nanpercentile(focal_grid["pm25"], 99.5)) * 1.1

fig, ax = plt.subplots(figsize=(13, 4.8))
A.shade_aqi_bands(ax, ymax=ymax)
for dev, c in zip(FOCAL, sns.color_palette("deep", len(FOCAL))):
    d = focal_grid[focal_grid["device_name"] == dev].sort_values("datetime")
    obs_d = d[d["is_observed"]]
    ax.plot(obs_d["datetime"], obs_d["pm25"], marker="o", ms=2.2, lw=0.8,
            color=c, label=dev, alpha=0.9)
ax.set_ylim(0, ymax)
ax.set_ylabel("PM2.5 (ug/m3)")
ax.set_xlabel("2026 (Kampala time)")
ax.legend(ncol=3, fontsize=7, loc="upper right")
ax.set_title("Fig 2.3  Hourly PM2.5 for the five focal sensors, with AQI bands "
             "(gaps = sensor offline)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
fig.tight_layout()
A.savefig(fig, "fig2_3_timeseries_aqi.png")

jan = obs[(obs["datetime"].dt.month == 1) & obs["device_name"].isin(FOCAL)]["pm25"].mean()
apr = obs[(obs["datetime"].dt.month == 4) & obs["device_name"].isin(FOCAL)]["pm25"].mean()
jun = obs[(obs["datetime"].dt.month == 6) & obs["device_name"].isin(FOCAL)]["pm25"].mean()
print(f"Focal mean PM2.5 by window: January {jan:.1f} | April {apr:.1f} | June {jun:.1f} ug/m3")
print(f"January is roughly {jan/max(apr,1e-9):.1f}x the April/June level - a seasonal")
print("level shift. DECISION: the model needs a `month` feature, and validation must")
print("include every window (see Step 3).")
print("saved fig2_3_timeseries_aqi.png")


 3. TIME SERIES WITH AQI BANDS


Focal mean PM2.5 by window: January 84.0 | April 34.1 | June 29.9 ug/m3
January is roughly 2.5x the April/June level - a seasonal
level shift. DECISION: the model needs a `month` feature, and validation must
include every window (see Step 3).
saved fig2_3_timeseries_aqi.png


## 4. Diurnal cycle
Mean PM2.5 by hour of day. The double peak is the reason hour-of-day is an essential feature, encoded as sine/cosine so that 23:00 and 00:00 are neighbours.

In [6]:
print("\n" + "="*78)
print(" 4. DIURNAL CYCLE")
print("="*78)

fig, ax = plt.subplots(figsize=(10, 4.5))
pooled = obs[obs["device_name"].isin(FOCAL)].copy()
diurnal_all = pooled.groupby("hour")["pm25"].mean()
for dev, c in zip(FOCAL, sns.color_palette("deep", len(FOCAL))):
    d = pooled[pooled["device_name"] == dev]
    ax.plot(d.groupby("hour")["pm25"].mean().index,
            d.groupby("hour")["pm25"].mean().values, marker="o", ms=3,
            lw=1.2, color=c, label=dev, alpha=0.85)
ax.plot(diurnal_all.index, diurnal_all.values, color="black", lw=2.4,
        marker="s", ms=3, label="pooled mean")
ax.set_xlabel("hour of day (Kampala time)")
ax.set_ylabel("mean PM2.5 (ug/m3)")
ax.set_xticks(range(0, 24))
ax.legend(fontsize=7)
ax.set_title("Fig 2.4  Mean PM2.5 by hour of day - a morning and an evening peak")
ax.grid(alpha=0.3)
fig.tight_layout()
A.savefig(fig, "fig2_4_diurnal.png")

peak_hours = diurnal_all.sort_values(ascending=False).head(4).index.tolist()
print(f"Highest-mean hours (pooled): {sorted(peak_hours)}")
print("Two peaks (morning and evening) with an early-afternoon minimum. Emissions")
print("(traffic, charcoal cooking) combine with a shallow, stable boundary layer at")
print("night and early morning; the afternoon sun deepens the mixing layer and")
print("dilutes PM2.5. DECISION: hour of day is an essential feature, encoded as a")
print("sine/cosine pair so 23:00 and 00:00 are treated as neighbours.")


 4. DIURNAL CYCLE


Highest-mean hours (pooled): [7, 8, 21, 22]
Two peaks (morning and evening) with an early-afternoon minimum. Emissions
(traffic, charcoal cooking) combine with a shallow, stable boundary layer at
night and early morning; the afternoon sun deepens the mixing layer and
dilutes PM2.5. DECISION: hour of day is an essential feature, encoded as a
sine/cosine pair so 23:00 and 00:00 are treated as neighbours.


## 5. Calendar plot
Daily mean PM2.5 coloured on the AQI scale. It answers 'which days were safe?' at a glance - and makes the data gaps impossible to miss.

In [7]:
print("\n" + "="*78)
print(" 5. CALENDAR PLOT (daily mean, AQI coloured)")
print("="*78)

daily = (obs[obs["device_name"].isin(FOCAL)]
         .groupby(obs["datetime"].dt.date)["pm25"].mean())
daily.index = pd.to_datetime(daily.index)
daily.name = "daily_mean"

def aqi_colour(v):
    for i, (lo, hi, label) in enumerate(A.AQI_BANDS):
        if lo <= v < (hi if np.isfinite(hi) else 1e9):
            return A.AQI_COLORS[i]
    return A.AQI_COLORS[-1]

fig, ax = plt.subplots(figsize=(13, 3.2))
all_days = pd.date_range("2026-01-01", "2026-07-31", freq="D")
for day in all_days:
    v = daily.get(day, np.nan)
    colour = "#dddddd" if not np.isfinite(v) else aqi_colour(v)
    ax.bar(day, 1, width=0.85, color=colour, edgecolor="white", linewidth=0.3)
ax.set_yticks([])
ax.set_xlabel("2026 (Kampala time)")
ax.set_title("Fig 2.5  Daily mean PM2.5 calendar (five focal sensors); grey = no data")
ax.legend(handles=A.aqi_legend_handles(), ncol=6, fontsize=6.5,
          loc="upper center", bbox_to_anchor=(0.5, -0.35))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
fig.tight_layout()
A.savefig(fig, "fig2_5_calendar.png")
print("Only the snapshot days are coloured; everything grey had no reading.")
print("saved fig2_5_calendar.png")


 5. CALENDAR PLOT (daily mean, AQI coloured)


Only the snapshot days are coloured; everything grey had no reading.
saved fig2_5_calendar.png


## 6. The Z-score transform
The brief asks for the Z-score of PM2.5 as the modelling target. We fit mu and sigma on all observed readings and store them in `z_params.json`; the inverse transform lives in `aq_common.inverse_z`, so no notebook re-derives it.

In [8]:
print("\n" + "="*78)
print(" 6. Z-SCORE TRANSFORM (the modelling target)")
print("="*78)

# Fit mu/sigma on ALL observed PM2.5 (the network), not just the focal five.
ZP = A.zscore_params(obs["pm25"])
A.save_zparams(ZP)
print(f"Global PM2.5: mean mu = {ZP['mean']:.2f} ug/m3, sd sigma = {ZP['std']:.2f} ug/m3 "
      f"(n = {ZP['n']:,})")
print("Target for every model:  z = (PM2.5 - mu) / sigma.  Inverse: PM2.5 = z*sigma + mu.")
print("The Z-score does not change the shape or ranking of values, only the scale -")
print("it helps distance-based models (KNN, SVR) and makes errors comparable.")

z_table = pd.DataFrame({"z": [-1, 0, 1, 2, 3, 5]})
z_table["PM2.5"] = (z_table["z"] * ZP["std"] + ZP["mean"]).round(1)
z_table["AQI"] = A.aqi_categorise(z_table["PM2.5"]).astype(str)
print("\n" + z_table.to_string(index=False))
z_table.to_csv(A.stage_dir("eda") / "zscore_table.csv", index=False)

edges_z = [(e - ZP["mean"]) / ZP["std"] for e in A.aqi_bins()[:-1]]
print("\nAQI band thresholds in z units:",
      ", ".join(f"{l}: z>={z:.2f}" for l, z in zip(A.AQI_LABELS[1:], edges_z[1:])))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].hist(obs["pm25"].dropna(), bins=60, color="#4c72b0", edgecolor="w", lw=0.3)
axes[0].axvline(ZP["mean"], color="#c44e52", ls="--", label=f"mean {ZP['mean']:.1f}")
axes[0].set_xlabel("PM2.5 (ug/m3)"); axes[0].set_ylabel("hours")
axes[0].set_title("raw PM2.5"); axes[0].legend(fontsize=7)
z = A.forward_z(obs["pm25"].dropna(), ZP)
axes[1].hist(z, bins=60, color="#55a868", edgecolor="w", lw=0.3)
axes[1].axvline(0, color="#c44e52", ls="--")
axes[1].set_xlabel("z-score"); axes[1].set_ylabel("hours")
axes[1].set_title("Z-scored (same shape, new scale)")
fig.suptitle("Fig 2.6  PM2.5 before and after Z-scoring", fontsize=12)
fig.tight_layout()
A.savefig(fig, "fig2_6_zscore.png")
print("saved fig2_6_zscore.png")


 6. Z-SCORE TRANSFORM (the modelling target)
Global PM2.5: mean mu = 32.53 ug/m3, sd sigma = 27.86 ug/m3 (n = 14,272)
Target for every model:  z = (PM2.5 - mu) / sigma.  Inverse: PM2.5 = z*sigma + mu.
The Z-score does not change the shape or ranking of values, only the scale -
it helps distance-based models (KNN, SVR) and makes errors comparable.

 z  PM2.5            AQI
-1    4.7           Good
 0   32.5       Moderate
 1   60.4      Unhealthy
 2   88.3      Unhealthy
 3  116.1      Unhealthy
 5  171.8 Very Unhealthy

AQI band thresholds in z units: Moderate: z>=-0.84, Unhealthy for Sensitive Groups: z>=0.11, Unhealthy: z>=0.82, Very Unhealthy: z>=3.34, Hazardous: z>=6.93


saved fig2_6_zscore.png


## 7. Weather and PM10 correlations
Pearson measures a straight-line association; Spearman measures monotonic co-movement and is safer for skewed data. The decisive observation: **PM10 is measured at the same instant as PM2.5**, so it would leak the answer and is not available when forecasting - it is excluded from the model.

In [9]:
print("\n" + "="*78)
print(" 7. CORRELATION OF TEMPERATURE / HUMIDITY / PM10 WITH PM2.5")
print("="*78)

rows = []
for dev in FOCAL:
    d = obs[obs["device_name"] == dev]
    for col in ("temp", "humidity"):
        pair = d[["pm25", col]].dropna()
        if len(pair) > 5:
            rows.append({"device_name": dev, "variable": col, "n": len(pair),
                         "pearson": pair["pm25"].corr(pair[col]),
                         "spearman": pair["pm25"].corr(pair[col], method="spearman")})
pool = obs[obs["device_name"].isin(FOCAL)]
for col in ("temp", "humidity", "pm10"):
    pair = pool[["pm25", col]].dropna()
    if len(pair) > 5:
        rows.append({"device_name": "ALL_5_POOLED", "variable": col, "n": len(pair),
                     "pearson": pair["pm25"].corr(pair[col]),
                     "spearman": pair["pm25"].corr(pair[col], method="spearman")})
corr = pd.DataFrame(rows).round(3)
print(corr.to_string(index=False))
corr.to_csv(A.stage_dir("eda") / "correlations.csv", index=False)
print("\nSpearman is preferred for a skewed variable: it measures monotonic co-movement")
print("without assuming a straight line. Correlation magnitude: 0.1-0.3 weak, 0.3-0.5")
print("moderate, >0.5 strong. Weather alone is at best moderately related to PM2.5, so")
print("it must be combined with time of day and site. PM10 correlates ~0.9+ because it")
print("is measured at the same instant - using it as a predictor would be LEAKAGE, so")
print("it is excluded from the model (Step 3).")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
sns.heatmap(corr.pivot_table(index="device_name", columns="variable",
                             values="spearman"), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=axes[0], cbar_kws={"label": "Spearman rho"})
axes[0].set_title("Spearman correlation with PM2.5")
for ax, col, lab in [(axes[1], "temp", "temperature (degC)")]:
    d = pool[["pm25", col]].dropna()
    ax.scatter(d[col], d["pm25"], s=5, alpha=0.25, color="#4c72b0")
    ax.set_xlabel(lab); ax.set_ylabel("PM2.5 (ug/m3)")
    ax.set_title("PM2.5 vs temperature (pooled)")
fig.suptitle("Fig 2.7  Weather-PM2.5 relationships", fontsize=12)
fig.tight_layout()
A.savefig(fig, "fig2_7_correlation.png")
print("saved fig2_7_correlation.png")


 7. CORRELATION OF TEMPERATURE / HUMIDITY / PM10 WITH PM2.5
    device_name variable   n  pearson  spearman
          aq_66     temp 147   -0.133    -0.130
          aq_66 humidity 147    0.155     0.256
       aq_g5_37     temp 147   -0.072    -0.088
       aq_g5_37 humidity 147   -0.031     0.038
          aq_47     temp  67   -0.247    -0.169
          aq_47 humidity  67    0.260     0.282
          aq_58     temp 145   -0.169    -0.219
          aq_58 humidity 145    0.252     0.273
ag_588c81267534     temp 142   -0.508    -0.558
ag_588c81267534 humidity 142    0.411     0.477
   ALL_5_POOLED     temp 648   -0.152    -0.153
   ALL_5_POOLED humidity 648    0.085     0.153
   ALL_5_POOLED     pm10 726    0.960     0.974

Spearman is preferred for a skewed variable: it measures monotonic co-movement
without assuming a straight line. Correlation magnitude: 0.1-0.3 weak, 0.3-0.5
moderate, >0.5 strong. Weather alone is at best moderately related to PM2.5, so
it must be combined with tim

saved fig2_7_correlation.png


## 8. Write the EDA decisions
These seven decisions are the contract between EDA and modelling. Step 3 reads them from `eda_meta.json` so the two notebooks cannot disagree.

In [10]:
print("\n" + "="*78)
print(" 8. EDA DECISIONS (carried into the model notebook)")
print("="*78)

DECISIONS = [
    "Target is the global Z-score of PM2.5: z = (PM2.5 - mu)/sigma.",
    "Keep the right tail (real pollution), so prefer tree models; report MAE and RMSE.",
    "Flag, do not delete, statistical outliers (traffic/cooking peaks are real).",
    "hour of day enters as sin/cos; month captures the Jan vs Apr/Jun level shift.",
    "device_name is the strongest signal (site baseline) - target-encode it, leak-free.",
    "PM10 is excluded from the model: same-instant measurement, not available in future.",
    "Weather (temp, humidity) is weakly-to-moderately correlated; use it but expect little.",
]
for i, d in enumerate(DECISIONS, 1):
    print(f"  {i}. {d}")

eda_meta = {
    "n_observed": int(len(obs)),
    "n_sensors": int(obs["device_name"].nunique()),
    "focal_sensors": FOCAL,
    "focal_table": focal.to_dict(orient="records"),
    "pooled_pm25": {
        "mean": float(obs["pm25"].mean()),
        "median": float(obs["pm25"].median()),
        "std": float(obs["pm25"].std()),
        "skewness": float(pooled_skew),
        "p95": float(obs["pm25"].quantile(0.95)),
        "max": float(obs["pm25"].max()),
    },
    "monthly_mean_pm25_focal": {"Jan": float(jan), "Apr": float(apr), "Jun": float(jun)},
    "zscore": ZP,
    "aqi_bands": A.AQI_BANDS,
    "decisions": DECISIONS,
    "correlations_pooled": corr[corr["device_name"] == "ALL_5_POOLED"]
        .drop(columns="device_name").to_dict(orient="records"),
    "env": A.env_facts(),
}
A.save_json(eda_meta, A.stage_dir("eda") / "eda_meta.json")
LOG.log("Wrote eda_meta.json and focal_sensors.csv")
print("\nsaved eda_meta.json")

summary = LOG.save({"stage": "EDA", "focal": FOCAL, "zscore": ZP})
print("\n" + "="*78)
print("  STEP 2 COMPLETE")
print("="*78)
for f in sorted(A.stage_dir("eda").glob("*")):
    print(f"  {f.name:32s} {f.stat().st_size:>10,d} bytes")
print("\nNext:  Step3_Modeling.ipynb")


 8. EDA DECISIONS (carried into the model notebook)
  1. Target is the global Z-score of PM2.5: z = (PM2.5 - mu)/sigma.
  2. Keep the right tail (real pollution), so prefer tree models; report MAE and RMSE.
  3. Flag, do not delete, statistical outliers (traffic/cooking peaks are real).
  4. hour of day enters as sin/cos; month captures the Jan vs Apr/Jun level shift.
  5. device_name is the strongest signal (site baseline) - target-encode it, leak-free.
  6. PM10 is excluded from the model: same-instant measurement, not available in future.
  7. Weather (temp, humidity) is weakly-to-moderately correlated; use it but expect little.
2026-10-18 12:22:15 | INFO | Wrote eda_meta.json and focal_sensors.csv

saved eda_meta.json

  STEP 2 COMPLETE
  correlations.csv                        484 bytes
  distribution_summary.csv                549 bytes
  eda_meta.json                         3,396 bytes
  fig2_1_distributions.png            302,118 bytes
  fig2_2_boxplots.png                  4